In [1]:
import cv2

In [2]:
import cv2
import os

def generate_dataset(person_name):
    face_classifier = cv2.CascadeClassifier("C:/Users/aksha/OneDrive/Desktop/demo2/haarcascade_frontalface_default.xml")
    save_path = os.path.join("C:/Users/aksha/OneDrive/Desktop/demo2/data/", person_name)
    
    # Create a directory for the person if it doesn't exist
    os.makedirs(save_path, exist_ok=True)
    
    img_id = 0

    def face_cropped(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_classifier.detectMultiScale(gray, 1.3, 5)
        
        if len(faces) == 0:
            return None
        for (x, y, w, h) in faces:
            cropped_face = img[y:y+h, x:x+w]
        return cropped_face

    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

    while True:
        ret, frame = cap.read()
        if face_cropped(frame) is not None and img_id < 200:
            img_id += 1
            face = cv2.resize(face_cropped(frame), (200, 200))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            # Save images with the format person_name.img_id.jpg
            file_name_path = os.path.join(save_path, f"{person_name}.{img_id}.jpg")
            cv2.imwrite(file_name_path, face)
            cv2.putText(face, str(img_id), (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow("Cropped face", face)

        # Stop if 200 images are captured or Enter is pressed
        if cv2.waitKey(1) == 13 or img_id >= 200:
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"Collecting samples for {person_name} is completed.")

# Capture images for a new person by specifying their name
generate_dataset("Saurabh")


In [3]:
import os
import cv2
from PIL import Image  # pip install pillow
import numpy as np

def train_classifier(data_dir):
    faces = []
    ids = []
    label_dict = {}
    label_id = 0

    # Iterate over each person folder in the dataset directory
    for person_name in os.listdir(data_dir):
        person_path = os.path.join(data_dir, person_name)
        
        if os.path.isdir(person_path):
            label_dict[label_id] = person_name  # Map label to person name
            
            # Process each image in the person's folder
            for image_file in os.listdir(person_path):
                img_path = os.path.join(person_path, image_file)
                img = Image.open(img_path).convert('L')
                imageNp = np.array(img, 'uint8')
                
                faces.append(imageNp)
                ids.append(label_id)
            
            label_id += 1  # Move to the next label for the next person
    
    ids = np.array(ids)

    # Train and save classifier
    clf = cv2.face.LBPHFaceRecognizer_create()
    clf.train(faces, ids)
    clf.write("C:/Users/aksha/OneDrive/Desktop/demo2/classifier.xml")

    # Save label dictionary to reference names later
    np.save("C:/Users/aksha/OneDrive/Desktop/demo2/label_dict.npy", label_dict)

train_classifier("C:/Users/aksha/OneDrive/Desktop/demo2/data/")


In [5]:
import cv2
import numpy as np

def draw_boundary(img, classifier, scaleFactor, minNeighbors, color, clf, labels):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_img = cv2.equalizeHist(gray_img)
    
    features = classifier.detectMultiScale(gray_img, scaleFactor, minNeighbors)
    
    for (x, y, w, h) in features:
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        
        # Predict the identity of the person
        id, pred = clf.predict(gray_img[y:y + h, x:x + w])
        confidence = int(100 * (1 - pred / 300))
        
        # Map ID to name if confidence is above a threshold
        name = labels.get(id, "UNKNOWN")
        
        if confidence > 75:
            display_text = f"{name} ({confidence}%)"
        else:
            display_text = "UNKNOWN"
        
        # Display name and confidence
        cv2.putText(img, display_text, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    
    return img

# Load face detector and classifier with trained parameters
faceCascade = cv2.CascadeClassifier("C:/Users/aksha/OneDrive/Desktop/demo2/haarcascade_frontalface_default.xml")
clf = cv2.face.LBPHFaceRecognizer_create(radius=2, neighbors=8, grid_x=8, grid_y=8)
clf.read("C:/Users/aksha/OneDrive/Desktop/demo2/classifier.xml")

# Load label dictionary for name mapping
labels = np.load("C:/Users/aksha/OneDrive/Desktop/demo2/label_dict.npy", allow_pickle=True).item()

# Capture video from webcam
video_capture = cv2.VideoCapture(0, cv2.CAP_DSHOW)

while True:
    ret, img = video_capture.read()
    img = draw_boundary(img, faceCascade, 1.3, 6, (255, 255, 255), clf, labels)
    cv2.imshow("Face Detection", img)

    if cv2.waitKey(1) == 13:  # Press Enter key to exit
        break

video_capture.release()
cv2.destroyAllWindows()
